### Topological data analysis

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
import gudhi as gd
import umap.umap_ as umap

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------
base_dir = "/home/a/projects/Complete-Neural-Signal-Analysis"

embedding_3d_dir = os.path.join(base_dir, "embedding_data", "3dembedding_data")
reuse_umap_dir = os.path.join(base_dir, "embedding_data", "tsne_umap_from_3d")

topology_dir = os.path.join(base_dir, "topology_from_3d_umap")
umap_cache_dir = os.path.join(topology_dir, "umap_embeddings")
persistence_dir = os.path.join(topology_dir, "persistence_diagrams")

betti_plot_dir = os.path.join(base_dir, "plots", "betti_plots_from_3d_umap")
persistence_plot_dir = os.path.join(base_dir, "plots", "persistence_plots_from_3d_umap")

for d in [topology_dir, umap_cache_dir, persistence_dir, betti_plot_dir, persistence_plot_dir]:
    os.makedirs(d, exist_ok=True)

eeg_channel_names = [
    'Fp1', 'Fpz', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2', 'FC6',
    'M1', 'T7', 'C3', 'Cz', 'C4', 'T8', 'M2', 'CP5', 'CP1', 'CP2', 'CP6',
    'P7', 'P3', 'Pz', 'P4', 'P8', 'POz', 'O1', 'Oz', 'O2'
]

# UMAP parameters
n_components_umap = 2
n_neighbors_umap = 15
min_dist_umap = 0.1
random_state_umap = 42

# Persistence settings
# Full Rips on all ~100k points is not feasible, so topology uses a fixed subset
persistence_max_points = 2500
rips_max_edge_length = 2.0
rips_max_dimension = 2

# Synthetic labels for demonstration classifier
segment_length = 1000
test_size = 0.30
random_state_split = 42
rf_estimators = 200
rf_random_state = 42

# Plot style
ACCENT = "cyan"
BG = "black"

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": ACCENT,
    "axes.labelcolor": ACCENT,
    "axes.edgecolor": ACCENT,
    "axes.titlecolor": ACCENT,
    "xtick.color": ACCENT,
    "ytick.color": ACCENT,
    "grid.color": ACCENT,
    "legend.facecolor": BG,
    "legend.edgecolor": ACCENT,
})

rng = np.random.default_rng(42)

# -------------------------------------------------------
# STYLE HELPER
# -------------------------------------------------------
def style_ax(ax):
    ax.set_facecolor(BG)
    ax.grid(True, alpha=0.20, color=ACCENT)
    ax.tick_params(colors=ACCENT)
    for spine in ax.spines.values():
        spine.set_color(ACCENT)

# -------------------------------------------------------
# BASIC HELPERS
# -------------------------------------------------------
def standardize_columns(X):
    X = np.asarray(X, dtype=float)
    mu = np.mean(X, axis=0, keepdims=True)
    sd = np.std(X, axis=0, keepdims=True) + 1e-12
    return (X - mu) / sd

def subsample_rows(X, max_points=None, rng=None):
    X = np.asarray(X)
    if rng is None:
        rng = np.random.default_rng()

    n = len(X)
    if max_points is None or n <= max_points:
        return X

    idx = np.sort(rng.choice(n, size=max_points, replace=False))
    return X[idx]

def make_synthetic_labels(n, segment_length=1000):
    num_segments = int(np.ceil(n / segment_length))
    labels = np.repeat(np.arange(num_segments) % 2, segment_length)[:n]
    return labels.astype(int)

def manual_train_test_split(X, y, test_size=0.3, random_state=42):
    X = np.asarray(X)
    y = np.asarray(y)
    n = len(X)

    rng_local = np.random.default_rng(random_state)
    perm = rng_local.permutation(n)

    n_test = int(np.round(test_size * n))
    test_idx = perm[:n_test]
    train_idx = perm[n_test:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def manual_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(y_true == y_pred))

# -------------------------------------------------------
# UMAP HELPERS
# -------------------------------------------------------
def apply_umap(data, n_components=2, n_neighbors=15, min_dist=0.1, random_state=42):
    reducer = umap.UMAP(
        n_components=n_components,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric="euclidean",
        random_state=random_state,
        transform_seed=random_state,
        low_memory=True
    )
    return reducer.fit_transform(data)

def load_or_compute_umap(channel_name, X_std):
    # first try the already existing UMAP embedding from your previous pipeline
    reuse_path = os.path.join(reuse_umap_dir, f"umap_embedding_{channel_name}.npy")
    if os.path.exists(reuse_path):
        emb = np.load(reuse_path)
        return emb, reuse_path, 0.0, True

    # then try local topology cache
    cache_path = os.path.join(umap_cache_dir, f"umap_embedding_{channel_name}.npy")
    if os.path.exists(cache_path):
        emb = np.load(cache_path)
        return emb, cache_path, 0.0, True

    # otherwise compute
    t0 = time.perf_counter()
    emb = apply_umap(
        X_std,
        n_components=n_components_umap,
        n_neighbors=n_neighbors_umap,
        min_dist=min_dist_umap,
        random_state=random_state_umap
    )
    elapsed = time.perf_counter() - t0

    np.save(cache_path, emb)
    return emb, cache_path, elapsed, False

# -------------------------------------------------------
# PERSISTENCE HELPERS
# -------------------------------------------------------
def compute_persistence(reduced_data):
    """
    Compute persistence on a reduced 2D cloud.
    """
    rips_complex = gd.RipsComplex(
        points=reduced_data,
        max_edge_length=rips_max_edge_length
    )
    simplex_tree = rips_complex.create_simplex_tree(max_dimension=rips_max_dimension)
    persistence = simplex_tree.persistence()

    intervals_by_dim = {}
    for dim in range(rips_max_dimension + 1):
        try:
            intervals_by_dim[dim] = simplex_tree.persistence_intervals_in_dimension(dim)
        except Exception:
            intervals_by_dim[dim] = np.empty((0, 2), dtype=float)

    try:
        betti = simplex_tree.betti_numbers()
    except Exception:
        betti = []

    betti = list(betti)
    if len(betti) < 3:
        betti = betti + [0] * (3 - len(betti))

    return persistence, intervals_by_dim, betti[:3]

def plot_persistence_diagram_manual(intervals_by_dim, channel_name, save_path):
    fig, ax = plt.subplots(figsize=(7, 6), facecolor=BG)
    style_ax(ax)

    all_finite_deaths = []
    for dim in intervals_by_dim:
        arr = np.asarray(intervals_by_dim[dim], dtype=float)
        if arr.size == 0:
            continue
        finite_mask = np.isfinite(arr[:, 1])
        if np.any(finite_mask):
            all_finite_deaths.extend(arr[finite_mask, 1].tolist())

    max_finite = max(all_finite_deaths) if len(all_finite_deaths) > 0 else 1.0
    inf_cap = max_finite * 1.05

    markers = {0: "o", 1: "s", 2: "^"}

    for dim in range(3):
        arr = np.asarray(intervals_by_dim.get(dim, np.empty((0, 2))), dtype=float)
        if arr.size == 0:
            continue

        births = arr[:, 0]
        deaths = arr[:, 1].copy()
        deaths[~np.isfinite(deaths)] = inf_cap

        ax.scatter(
            births, deaths,
            s=18,
            marker=markers.get(dim, "o"),
            facecolors="none",
            edgecolors=ACCENT,
            linewidths=1.0,
            label=f"H{dim}"
        )

    lim = max(inf_cap, 1e-6)
    ax.plot([0, lim], [0, lim], linestyle="--", color=ACCENT, alpha=0.8)

    ax.set_title(f"Persistence Diagram | {channel_name}")
    ax.set_xlabel("Birth")
    ax.set_ylabel("Death")
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()

def plot_betti_numbers(betti_numbers, channel_name, save_path):
    dims = [0, 1, 2]
    vals = list(betti_numbers)

    fig, ax = plt.subplots(figsize=(6, 4), facecolor=BG)
    style_ax(ax)
    ax.bar(dims, vals, color=ACCENT, edgecolor=ACCENT)
    ax.set_title(f"Betti Numbers | {channel_name}")
    ax.set_xlabel("Dimension")
    ax.set_ylabel("Betti Number")
    ax.set_xticks(dims)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()

# -------------------------------------------------------
# MAIN LOOP
# -------------------------------------------------------
summary_rows = []

for channel_name in eeg_channel_names:
    file_path = os.path.join(embedding_3d_dir, f"3dembedded_{channel_name}.npy")

    if not os.path.exists(file_path):
        print(f"Missing file for {channel_name}: {file_path}")
        continue

    channel_data_3d = np.load(file_path)
    channel_data_3d = np.asarray(channel_data_3d, dtype=float)

    if channel_data_3d.ndim != 2 or channel_data_3d.shape[1] != 3:
        print(f"Skipping {channel_name}: expected shape (N, 3), got {channel_data_3d.shape}")
        continue

    X3 = standardize_columns(channel_data_3d)

    print(f"\nProcessing {channel_name} | 3D shape={X3.shape}")

    # -------------------------
    # UMAP (reuse if already saved)
    # -------------------------
    reduced_data_umap, umap_path_used, umap_time, umap_reused = load_or_compute_umap(channel_name, X3)

    if reduced_data_umap.ndim != 2 or reduced_data_umap.shape[1] != 2:
        print(f"Skipping {channel_name}: bad UMAP shape {reduced_data_umap.shape}")
        continue

    # -------------------------
    # Topology subset
    # -------------------------
    reduced_topology = subsample_rows(reduced_data_umap, max_points=persistence_max_points, rng=rng)

    t0 = time.perf_counter()
    persistence, intervals_by_dim, betti_numbers = compute_persistence(reduced_topology)
    persistence_time = time.perf_counter() - t0

    # -------------------------
    # Save persistence data
    # -------------------------
    persistence_npz_path = os.path.join(persistence_dir, f"persistence_{channel_name}.npz")
    np.savez_compressed(
        persistence_npz_path,
        H0=np.asarray(intervals_by_dim.get(0, np.empty((0, 2))), dtype=float),
        H1=np.asarray(intervals_by_dim.get(1, np.empty((0, 2))), dtype=float),
        H2=np.asarray(intervals_by_dim.get(2, np.empty((0, 2))), dtype=float),
        betti=np.asarray(betti_numbers, dtype=int),
        n_points_topology=int(reduced_topology.shape[0]),
        channel=channel_name
    )

    # -------------------------
    # Plot persistence diagram
    # -------------------------
    persistence_plot_path = os.path.join(persistence_plot_dir, f"Persistence_{channel_name}.png")
    plot_persistence_diagram_manual(intervals_by_dim, channel_name, persistence_plot_path)

    # -------------------------
    # Plot Betti numbers
    # -------------------------
    betti_plot_path = os.path.join(betti_plot_dir, f"Betti_{channel_name}.png")
    plot_betti_numbers(betti_numbers, channel_name, betti_plot_path)

    # -------------------------
    # Synthetic labels
    # -------------------------
    labels = make_synthetic_labels(len(reduced_data_umap), segment_length=segment_length)

    # -------------------------
    # Manual split + RF classifier
    # -------------------------
    X_train, X_test, y_train, y_test = manual_train_test_split(
        reduced_data_umap,
        labels,
        test_size=test_size,
        random_state=random_state_split
    )

    clf = RandomForestClassifier(
        n_estimators=rf_estimators,
        random_state=rf_random_state,
        n_jobs=-1
    )
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    accuracy = manual_accuracy(y_test, y_pred)

    print(f"Betti numbers for {channel_name}: {tuple(betti_numbers)}")
    print(f"Model accuracy for {channel_name}: {accuracy:.4f}")

    # -------------------------
    # Summary row
    # -------------------------
    H0 = np.asarray(intervals_by_dim.get(0, np.empty((0, 2))), dtype=float)
    H1 = np.asarray(intervals_by_dim.get(1, np.empty((0, 2))), dtype=float)
    H2 = np.asarray(intervals_by_dim.get(2, np.empty((0, 2))), dtype=float)

    summary_rows.append({
        "channel": channel_name,
        "n_points_3d": X3.shape[0],
        "n_points_umap": reduced_data_umap.shape[0],
        "n_points_topology": reduced_topology.shape[0],
        "umap_reused": bool(umap_reused),
        "umap_time_sec": float(umap_time),
        "persistence_time_sec": float(persistence_time),
        "betti_0": int(betti_numbers[0]),
        "betti_1": int(betti_numbers[1]),
        "betti_2": int(betti_numbers[2]),
        "n_H0_intervals": int(len(H0)),
        "n_H1_intervals": int(len(H1)),
        "n_H2_intervals": int(len(H2)),
        "rf_accuracy": float(accuracy),
        "umap_source": os.path.basename(umap_path_used),
        "persistence_file": os.path.basename(persistence_npz_path),
    })

# -------------------------------------------------------
# SAVE SUMMARY
# -------------------------------------------------------
summary_df = pd.DataFrame(summary_rows)
summary_csv = os.path.join(topology_dir, "topology_summary_from_3d_umap.csv")
summary_txt = os.path.join(topology_dir, "topology_summary_from_3d_umap.txt")

summary_df.to_csv(summary_csv, index=False)

with open(summary_txt, "w") as f:
    f.write("Topology from 3D embeddings after UMAP\n")
    f.write("=====================================\n\n")
    f.write(f"base_dir: {base_dir}\n")
    f.write(f"embedding_3d_dir: {embedding_3d_dir}\n")
    f.write(f"reuse_umap_dir: {reuse_umap_dir}\n")
    f.write(f"persistence_max_points: {persistence_max_points}\n")
    f.write(f"rips_max_edge_length: {rips_max_edge_length}\n")
    f.write(f"rips_max_dimension: {rips_max_dimension}\n")
    f.write(f"segment_length: {segment_length}\n")
    f.write(f"test_size: {test_size}\n\n")

    if len(summary_df) > 0:
        f.write("Per-channel summary:\n")
        for _, row in summary_df.iterrows():
            f.write(
                f"{row['channel']}: "
                f"Betti=({row['betti_0']},{row['betti_1']},{row['betti_2']}), "
                f"H0/H1/H2=({row['n_H0_intervals']},{row['n_H1_intervals']},{row['n_H2_intervals']}), "
                f"RF_acc={row['rf_accuracy']:.4f}, "
                f"UMAP_reused={row['umap_reused']}, "
                f"UMAP_time={row['umap_time_sec']:.2f}s, "
                f"Persistence_time={row['persistence_time_sec']:.2f}s\n"
            )

print(f"\nSaved summary CSV: {summary_csv}")
print(f"Saved summary TXT: {summary_txt}")
print("Done.")